# Part 1: Tokenization and Vocabulary

## Notebook 3 — Lossy Compression: Tokenization, AE, and VAE

In Notebook 1 and 2 we saw that tokenization discards information to fit text into a fixed vocabulary. The same idea appears across machine learning under different names: autoencoders, VAEs, quantization. This notebook connects the dots.

- Tokenization is **discrete** lossy compression (integers)
- Autoencoders are **continuous** lossy compression (floats)
- A VAE adds structure so you can sample from the compressed space
- VQ-VAE bridges the two by quantizing the continuous latent into discrete codes
- We end with RT-1's action binning as a concrete example of naive discrete compression in robotics


### 1. Tokenization as Lossy Compression

A tokenizer takes text and maps it to a sequence of token IDs from a fixed vocabulary. Information is lost when:
- Whitespace and casing are stripped
- Rare words are split into subwords that lose the original spelling
- The decoder cannot perfectly reconstruct the original string

We saw this in Notebook 1: `'Hello World!'` → `[15496, 2159, 0]` → `'Hello World!'` (round-trip mostly works), but `'end-effector'` → `['end', '-', 'effect', 'or']` — the hyphen is now a separate token. The reconstruction is close but not exact.

All compression discards something. The question is what and how much.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Compression ratio: characters → tokens
text = "pick up the red cube and place it on the table"
tokens = tokenizer.encode(text)
print(f"Characters: {len(text)}")
print(f"Tokens:     {len(tokens)}")
print(f"Compression ratio: {len(text) / len(tokens):.1f} chars per token")
print(f"\nToken IDs: {tokens}")
print(f"Decoded:   {tokenizer.decode(tokens)}")

### 2. Autoencoders: Continuous Compression

An autoencoder (AE) compresses input through a low-dimensional bottleneck, then reconstructs it. Unlike a tokenizer, the latent z is continuous — any real number along the bottleneck dimension. Information is lost when the input must be squeezed through fewer dimensions than it originally had.

The AE is **self-supervised**: the input and the target are the same. The encoder compresses x → z, the decoder tries to reconstruct z → x̂, and the loss is the reconstruction error.


In [ ]:
import torch
import torch.nn as nn

# Toy data: circle of 2D points with noise
torch.manual_seed(42)
n = 500
theta = torch.rand(n) * 2 * torch.pi
radius = 1.0 + 0.1 * torch.randn(n)
data = torch.stack([radius * torch.cos(theta), radius * torch.sin(theta)], dim=1)

# Autoencoder (AE): 2D → 1D bottleneck → 2D
class AE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
            nn.Linear(16, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = AE(latent_dim=1)
opt = torch.optim.Adam(ae.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for step in range(500):
    opt.zero_grad()
    loss = loss_fn(ae(data), data)
    loss.backward()
    opt.step()

with torch.no_grad():
    z_ae = ae.encoder(data)
    recon_ae = ae(data)

print(f"AE reconstruction MSE: {loss_fn(recon_ae, data):.4f}")
print(f"Latent z range: [{z_ae.min():.2f}, {z_ae.max():.2f}]")
print(f"Latent z mean/std: {z_ae.mean():.3f} / {z_ae.std():.3f}")
print(f"The latent has no structure — values are scattered arbitrarily.")

An autoencoder and a tokenizer are two forms of the same idea: **lossy compression**. A tokenizer compresses text into a fixed vocabulary of discrete symbols, discarding information that does not fit the vocabulary. An autoencoder compresses data through a low-dimensional bottleneck, discarding what cannot be expressed in fewer dimensions.

The difference is that the AE's latent z is continuous (any real value along the bottleneck) while a tokenizer produces discrete integer IDs. To bridge the gap, you quantize the latent into discrete codes — that is what VQ-VAE does (section 4).


### 3. Variational Autoencoders

The AE's latent space has no structure — we cannot sample a valid z from it or interpolate between points. A Variational Autoencoder (VAE) fixes this.

Instead of outputting a point z, the VAE encoder outputs a **distribution**: a mean μ and log-variance log(σ²). We sample z = μ + σ·ε with ε ~ N(0,1) (the reparameterization trick).

The loss has two terms:
- **Reconstruction (MSE)**: how well the decoder reconstructs the input
- **KL divergence**: D_KL(q(z|x) || N(0,1)) — pushes the data-dependent posterior toward a fixed standard normal prior

The weight β balances them: loss = MSE + β · KL.


In [ ]:
# VAE: encoder outputs mu and log_var
# Loss = reconstruction_MSE + beta * KL( q(z|x) || N(0,1) )

class VAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
        )
        self.mu_head = nn.Linear(16, latent_dim)
        self.logvar_head = nn.Linear(16, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu_head(h), self.logvar_head(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

# Train VAEs with different KL weights
beta_values = [0.001, 0.05, 1.0]
trained_vaes = {}

for beta in beta_values:
    vae = VAE(latent_dim=1)
    opt = torch.optim.Adam(vae.parameters(), lr=0.01)
    for step in range(1000):
        opt.zero_grad()
        recon, mu, logvar = vae(data)
        recon_loss = loss_fn(recon, data)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / n
        loss = recon_loss + beta * kl_loss
        loss.backward()
        opt.step()
    trained_vaes[beta] = vae

# Compare AE vs VAE at different beta values
with torch.no_grad():
    print(f"{'Model':<16} {'β':>8} {'Recon MSE':>12} {'KL Div':>12} {'μ mean':>10} {'σ mean':>10}")
    print('─' * 72)
    print(f"{'AE (no KL)':<16} {'—':>8} {loss_fn(recon_ae, data).item():>12.4f} {'—':>12} {z_ae.mean().item():>10.3f} {z_ae.std().item():>10.3f}")
    for beta in beta_values:
        vae = trained_vaes[beta]
        recon, mu, logvar = vae(data)
        kl_value = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / n
        sigma = torch.exp(0.5 * logvar)
        print(f"{'VAE':<16} {beta:>8.3f} {loss_fn(recon, data).item():>12.4f} {kl_value.item():>12.4f} {mu.mean().item():>10.3f} {sigma.mean().item():>10.3f}")

# Generate from the prior: sample z ~ N(0,1), decode, check quality
print(f"\nGenerate by sampling z ~ N(0,1) and decoding:")
for beta in beta_values:
    vae = trained_vaes[beta]
    z_sample = torch.randn(100, 1)
    gen = vae.decoder(z_sample)
    avg_dist = torch.cdist(gen, data).min(dim=1).values.mean()
    print(f"  β={beta:.3f}: avg distance to nearest data = {avg_dist:.3f}")
print(f"\nLow β → good reconstruction, latent is unstructured.")
print(f"High β → latent ≈ N(0,1) (good for generation), poor reconstruction.")

### Why N(0,1)?

The prior is not arbitrary. When you want to **generate** new data with a VAE, you sample z ~ N(0,1) and run the decoder. If the latent distribution during training was unconstrained (like the AE), a random z from N(0,1) would land in a region the decoder never saw during training and produce garbage.

KL divergence solves this by pulling the encoder's output distribution toward N(0,1). The two forces:

| Term | What it does | Low β | High β |
|---|---|---|---|
| Reconstruction (MSE) | Push z to capture input details | Dominates: good recon, no structure | Weak: poor recon |
| KL divergence | Pull q(z\|x) toward N(0,1) | Weak: prior ignored | Dominates: latent ≈ N(0,1), hard to reconstruct |

In practice, β is a hyperparameter (typical range 0.001–0.1). A Conditional VAE (CVAE) adds observation conditioning — the encoder and decoder both receive an extra input like an image or robot state (we will see this in the ACT notebook, Part 2).


### 4. VQ-VAE: The Bridge to Discrete Tokens

A VAE compresses to continuous latents. A tokenizer compresses to discrete IDs. VQ-VAE (Vector Quantized VAE) sits in the middle.

Instead of sampling from a distribution, VQ-VAE finds the nearest vector in a learned codebook and passes that codebook index forward. The latent becomes a discrete integer — a token ID. The decoder reconstructs from the corresponding codebook vector.

This is exactly the pattern behind learned action tokenizers like FAST: compress a chunk of robot actions, quantize to discrete codes, then use those codes as tokens for a transformer. We will see this in Part 3.


### 5. RT-1: Naive Discrete Compression for Robot Actions

The simplest way to apply discrete compression to robot actions is per-dimension binning, used by RT-1 (Brohan et al., 2022). Each degree of freedom (x, y, z, roll, pitch, yaw, gripper) is an independent scalar, discretized into a fixed number of bins.

With 256 bins per dimension and a typical range of [-2, 2]:
- The continuous range is split into 256 equal-width bins (each ~0.016 wide)
- Each action step produces 7 tokens (one per dimension)
- An action chunk of 100 steps requires 700 tokens

This is lossy — small position differences get rounded to the same bin. And it is expensive: each dimension needs its own discrete vocabulary. In Part 3 we will see how FAST improves on this by operating in the frequency domain.


In [ ]:
import numpy as np

# A 7-DoF robot action: [x, y, z, roll, pitch, yaw, gripper]
# Example: [0.15, -0.02, 0.84, -0.12, 1.57, 0.03, 1.0]
action = np.random.uniform(-1.5, 1.5, size=7)

# RT-1 style: bin each dimension into 256 discrete values
n_bins = 256
bounds = (-2.0, 2.0)

# Normalize to [0, 1), then map to bin via floor
scaled = (action - bounds[0]) / (bounds[1] - bounds[0])
token_ids = np.floor(scaled * n_bins).astype(int)
token_ids = np.clip(token_ids, 0, n_bins - 1)

print("Action (7-DoF):")
print(f"  x={action[0]:.2f}  y={action[1]:.2f}  z={action[2]:.2f}  "
      f"roll={action[3]:.2f}  pitch={action[4]:.2f}  yaw={action[5]:.2f}  "
      f"gripper={action[6]:.2f}")
print(f"  Token IDs (0-255): {token_ids.tolist()}")
print(f"  Tokens per action step: {len(token_ids)}")

# For a 100-step chunk (same horizon as ACT)
chunk = 100
total_tokens = chunk * len(token_ids)
print(f"\nChunk of {chunk} steps: {total_tokens} tokens")
print(f"  (FAST tokenizer reduces this to ~45 via DCT + BPE)")

# Reconstruction: midpoint of each bin
bin_width = (bounds[1] - bounds[0]) / n_bins
reconstructed = bounds[0] + bin_width * (token_ids + 0.5)
error = np.abs(action - reconstructed)
print(f"\nReconstructed action:")
print(f"  x={reconstructed[0]:.4f}  y={reconstructed[1]:.4f}  z={reconstructed[2]:.4f}  "
      f"roll={reconstructed[3]:.4f}  pitch={reconstructed[4]:.4f}  yaw={reconstructed[5]:.4f}  "
      f"gripper={reconstructed[6]:.4f}")
print(f"  Abs error: {np.array2string(error, precision=4)}")
print(f"\nBin width: {bin_width:.4f}")
print(f"  Worst-case quantization error: {bin_width/2:.4f}")
print(f"  Actual max error: {error.max():.4f}")

### Where This Leaves Us

We have seen three forms of lossy compression:
- **Tokenization** (discrete): fixed vocabulary, integer outputs
- **Autoencoders** (continuous): learned bottleneck, float latents
- **RT-1 binning** (discrete, naive): uniform bins per dimension, 700 tokens per action chunk

In Part 2, we will see how five real VLAs represent robot actions — and notably, four of them avoid tokenization entirely. They use continuous representations with dedicated action experts. Then in Part 3, we will see FAST bring tokenization back via frequency-domain compression.
